In [168]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from metpy.calc import wind_speed
from metpy.calc import wind_direction 
from metpy.units import units

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'font.size' : 18})

# About
__Author:__ Pat McCornack  
__Date:__ 04/11/25  
__Purpose:__ Uses extracted netcdfs for standard meteorological variables and Chris Still met station data to evaluate WRF's performance.  

# Functions

# Load/Process Data

In [169]:
root_dir = Path().resolve().parents[0]
wrf_data_dir = root_dir / 'data' /'wrf'
met_df_fpath = root_dir /  'data' / 'weather-station' / 'combined-sites.csv'
met_coords_fpath = root_dir / 'data' / 'weather-station' / 'sci-stations.csv'

wrf_dict = {'RH2' : wrf_data_dir / 'wrf-rh2.nc',
            'T2' : wrf_data_dir / 'wrf-T2.nc',
            'U10' : wrf_data_dir / 'wrf-U10.nc',  # x wind component (10m)
            'V10' : wrf_data_dir / 'wrf-V10.nc'}  # y wind component (10m)

met_coords = pd.read_csv(met_coords_fpath)  # Info about the weather stations
met_coords = met_coords.loc[~met_coords['station'].isin(['eend', 'eenc', 'crat', 'pozo-smo2', 'pozo-smo1'])]  # Drop irrelevant sites

## Process Weather Station Data

In [170]:
# Set up met_df
met_df = pd.read_csv(met_df_fpath, index_col=0)
met_df['time (PST)'] = pd.to_datetime(met_df['time (PST)'])

met_df = met_df[['site', 'time (PST)', 'air temperature (C)', 'relative humidity (%)', 'wind speed (m/s)', 'wind direction (deg)']]
met_df = met_df.loc[~met_df['site'].isin(['eend', 'eenc', 'crat', 'pozo-smo2', 'pozo-smo1'])]  # Drop irrelevant sites

met_df.describe()


,time (PST),air temperature (C),relative humidity (%),wind speed (m/s),wind direction (deg)
count,622499,603912.000000,613093.000000,582516.000000,574226.000000
mean,2006-11-14 01:20:39.554279936,14.174732,69.607999,1.679329,225.004033
min,2003-12-19 16:00:00,0.850000,0.000000,0.000000,0.000000
25%,2005-05-06 22:15:00,10.780000,52.110000,0.540000,122.300000
50%,2006-05-21 02:00:00,13.150000,80.800000,1.432000,286.399990
75%,2008-05-15 03:07:30,16.799999,95.750000,2.555000,317.800000
max,2011-10-01 23:00:00,111.400000,100.000000,16.120000,677.000000
std,NaN,4.917584,31.357300,1.431682,112.578061


__Maximum outliers__  
Air temperatures maximum is 111.4 - which is impossible. There's several rows with this value, and these are all set to NAN.  
Similarly, there's a recorded wind direction value of over 360, this is also set to nan

In [171]:
# Handle outliers

# Air temperature over 50C is unlikely
met_df.loc[met_df['air temperature (C)'] > 50] # Values are all 111.4 -> set to NaN
met_df.loc[met_df['air temperature (C)'] > 50, 'air temperature (C)'] = np.nan

# Can't have wind direction over 360
met_df.loc[met_df['wind direction (deg)'] > 360, 'wind direction (deg)'] = np.nan

__Air Temperature Minimum Outliers__  
Ther's a bunch of air temperature observations from June 17, 2011 for Sauces Canyon. The low recorded by the RAWS stations for this day was 50*F, so these are probably bad. 

In [172]:
met_df.loc[(met_df['air temperature (C)'] < 2) & (met_df['time (PST)'].dt.month == 6), 'air temperature (C)'] = np.nan

In [173]:
met_df.describe()

,time (PST),air temperature (C),relative humidity (%),wind speed (m/s),wind direction (deg)
count,622499,603889.000000,613093.000000,582516.000000,574225.000000
mean,2006-11-14 01:20:39.554279936,14.174674,69.607999,1.679329,225.003246
min,2003-12-19 16:00:00,1.847000,0.000000,0.000000,0.000000
25%,2005-05-06 22:15:00,10.780000,52.110000,0.540000,122.300000
50%,2006-05-21 02:00:00,13.150000,80.800000,1.432000,286.399990
75%,2008-05-15 03:07:30,16.799999,95.750000,2.555000,317.800000
max,2011-10-01 23:00:00,40.670000,100.000000,16.120000,360.000000
std,NaN,4.912346,31.357300,1.431682,112.576579


In [174]:
# Set multiindex 
met_df = met_df.set_index(['site', 'time (PST)'])

In [181]:
met_df.loc[met_df.duplicated() == True]

air temperature (C)  relative humidity (%)  \
site time (PST)                                                        
wrdg 2004-04-05 14:45:00                  NaN                    0.0   
     2004-04-05 15:00:00                  NaN                    0.0   
     2004-04-05 15:15:00                  NaN                    0.0   
     2004-04-05 15:30:00                  NaN                    0.0   
     2004-04-05 15:45:00                  NaN                    0.0   
...                                       ...                    ...   
upem 2011-10-01 22:00:00                12.81                  100.0   
     2011-10-01 23:00:00                12.77                  100.0   
     2011-10-01 23:00:00                12.76                  100.0   
     2011-10-01 23:00:00                12.78                  100.0   
     2011-10-01 23:00:00                12.76                   99.9   

                          wind speed (m/s)  wind direction (deg)  
site time (PST)                                                   
wrdg 2004-04-05 14:45:00               0.0                   0.0  
     2004-04-05 15:00:00               0.0                   0.0  
     2004-04-05 15:15:00               0.0                   0.0  
     2004-04-05 15:30:00               0.0                   0.0  
     2004-04-05 15:45:00               0.0                   0.0  
...                                    ...                   ...  
upem 2011-10-01 22:00:00               NaN                   NaN  
     2011-10-01 23:00:00               NaN                   NaN  
     2011-10-01 23:00:00               NaN                   NaN  
     2011-10-01 23:00:00               NaN                   NaN  
     2011-10-01 23:00:00               NaN                   NaN  

[47399 rows x 4 columns]

## Process WRF data

In [175]:
# Extract dataframe from netcdfs using site coords
first_pass = True
for var, fpath in wrf_dict.items():  # For each variable
    var_df = pd.DataFrame()
    for i, row in met_coords.iterrows():  # For each station
        lat = row['latitude']
        lon = row['longitude']
        ds = xr.open_dataarray(wrf_dict[var])
        station_ds = ds.sel(lat=lat, lon=lon, method='nearest')
        station_df = station_ds.to_dataframe(name=var).reset_index()[['time', var]]
        station_df['site'] = row['station']

        station_df['time'] = pd.to_datetime(station_df['time'])
        station_df['time'] = station_df['time'] - pd.to_timedelta(1, unit='h')  # PDT to PST
        station_df = station_df.rename({'time' : 'time (PST)'}, axis=1)
        station_df = station_df.set_index(['site', 'time (PST)'])

        var_df = pd.concat([var_df, station_df])

    # Create single df with all variables
    if first_pass == True:
        wrf_df = var_df
    else:
        wrf_df = pd.merge(wrf_df, var_df, how='inner', left_index=True, right_index=True)
    
    first_pass = False

wrf_df.describe()


,RH2,T2,U10,V10
count,585179.000000,585179.000000,585179.000000,585179.000000
mean,69.397964,290.633636,2.342463,-1.729972
std,22.255562,5.408674,2.931606,2.346412
min,2.387302,0.000000,-15.134946,-12.375834
25%,54.739790,286.929840,0.546338,-3.280115
50%,71.825264,289.977692,2.611226,-1.616169
75%,87.442387,293.718628,4.293668,-0.178745
max,100.000000,313.371521,14.015853,7.244581


In [176]:
# Convert K to C
wrf_df['T2'] = wrf_df['T2'] - 273
wrf_df.loc[wrf_df['T2'] == -273] = np.nan  # Handle bad values

wrf_df['T2'].describe()

count    585144.000000
mean         17.650930
std           4.919634
min           5.605377
25%          13.930588
50%          16.978027
75%          20.719032
max          40.371521
Name: T2, dtype: float64

In [177]:
# Get wind speed/direction from components
u = wrf_df['U10'].values * units('m/s')
v = wrf_df['V10'].values * units('m/s')
wrf_df['wind_spd'] = wind_speed(u,v)
wrf_df['wind_dir'] = wind_direction(u, v, convention='from')
wrf_df = wrf_df.drop(['U10', 'V10'], axis=1)

wrf_df[['wind_spd', 'wind_dir']].describe()

,wind_spd,wind_dir
count,585144.000000,585144.000000
mean,4.094111,258.236786
std,2.411633,83.209549
min,0.007292,0.002151
25%,2.316300,227.048431
50%,3.611422,296.751709
75%,5.553937,311.518951
max,16.061771,359.999207


In [ ]:
# Rename WRF columns
new_cols = []
for col in list(wrf_df.columns):
    new_cols.append(col + '_wrf')

wrf_df.columns = new_cols
wrf_df.head()

RH2_wrf     T2_wrf  wind_spd_wrf  wind_dir_wrf
site time (PST)                                                           
cair 1996-05-31 16:00:00  67.323349  16.843048      5.707997    292.702515
     1996-05-31 17:00:00  70.684212  16.279541      5.039712    293.075470
     1996-05-31 18:00:00  73.011337  15.763458      4.491889    293.207214
     1996-05-31 19:00:00  76.270844  14.929565      3.399316    290.956421
     1996-05-31 20:00:00  80.533516  14.153656      2.519886    288.491150

In [179]:
# Join dataframes
join_df = pd.concat([met_df, wrf_df], join='inner', axis=1)
join_df.head()

InvalidIndexError: Reindexing only valid with uniquely valued Index objects